# Lab 04 — Probabilistic Models

**Guide:** `practice/04_probabilistic_models.pdf` explains every task. Read it alongside.

Cells marked **TODO** are yours: replace `raise NotImplementedError` with your code. Every other cell is ready and
draws the plots and prints the comparisons once your functions work. Run the notebook from top to bottom.

**The rule of this lab:** where a model needs an optimizer, it is *a negative log-likelihood written with `jax.numpy`*
and the gradient comes from `jax.grad` (through `train`). Naive Bayes needs no gradient: it is counting, and JAX scores
all examples at once, in log space. Every model is checked against its scikit-learn twin.

| Part | Task | twin | Time |
|:--|:--|:--|:-:|
| A | MLE and MAP with `jax.grad`: Gaussian, Beta prior, ridge as MAP | `Ridge`, `BernoulliNB` (`alpha`) | 20 min |
| B | Naive Bayes, **discrete** features: Bernoulli, Laplace smoothing | `BernoulliNB` | 15 min |
| C | Naive Bayes, **continuous** features: Gaussian; categorical (binned) | `GaussianNB`, `CategoricalNB` | 20 min |
| D | Naive Bayes for **regression**: binned target; continuous target | `GaussianNB`, `CategoricalNB`, `LinearRegression` | 35 min |
| E | Probabilities: log-loss, Brier, reliability; decisions with costs | `log_loss`, `brier_score_loss`, `calibration_curve` | 20 min |
| F | Challenge: Platt scaling with `jax.grad` | `LogisticRegression` | home |

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from jax.scipy.special import logsumexp
from lab04_utils import (
    apply_bins,
    classification_metrics,
    compare,
    fit_bins,
    fit_standardizer,
    load_housing,
    load_spam,
    plot_beta_posteriors,
    plot_cost_curve,
    plot_learning_curves,
    plot_pred_vs_actual,
    plot_reliability,
    regression_metrics,
    standardize,
    stratified_split,
    train,
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.naive_bayes import BernoulliNB, CategoricalNB, GaussianNB
from sklearn.preprocessing import KBinsDiscretizer

plt.rcParams["figure.figsize"] = (9, 4)
rng = np.random.default_rng(0)

## The data

* **Spam** (classification): `X_s` raw frequencies; `B_s` = "the word appears" (binary features for Bernoulli NB);
  `L_s` = $\log(1+x)$ (continuous features for Gaussian NB); `Z_s` = 4 quantile bins of `L_s` (categorical features).
* **House prices** (regression): standardized features, target in units of 100 000 USD.

Bin edges and scaling statistics are fitted on the **training** part only.

In [ ]:
X_s, y_s, names_s = load_spam()
X_h, y_h, names_h = load_housing()

tr_s, te_s = stratified_split(y_s, 0.2, rng)
ytr_s, yte_s = y_s[tr_s], y_s[te_s]
B_s, L_s = (X_s > 0).astype(float), np.log1p(X_s)
edges_s = fit_bins(L_s[tr_s], 4)
Z_s = apply_bins(L_s, edges_s)

perm = rng.permutation(len(y_h))
te_h, tr_h = perm[: len(y_h) // 5], perm[len(y_h) // 5 :]
mean_h, std_h = fit_standardizer(X_h[tr_h])
Xtr_h, Xte_h = standardize(X_h[tr_h], mean_h, std_h), standardize(X_h[te_h], mean_h, std_h)
ytr_h, yte_h = y_h[tr_h], y_h[te_h]
print("spam train/test:", len(tr_s), len(te_s), "| housing train/test:", len(tr_h), len(te_h))

## Part A — MLE and MAP with `jax.grad`

### A1. Gaussian maximum likelihood

Implement `gaussian_nll(params, x)`, the negative log-likelihood of $x_1,\dots,x_n\sim\mathcal{N}(\mu,\sigma^2)$ with
`params = {"mu": ..., "log_sigma": ...}` (optimizing $\log\sigma$ keeps $\sigma > 0$):

$$-\ell(\mu,\sigma) = \frac n2\log(2\pi\sigma^2) + \frac{1}{2\sigma^2}\sum_i (x_i-\mu)^2$$

In [ ]:
def gaussian_nll(params, x) -> jax.Array:
    """TODO: n/2 log(2 pi sigma^2) + sum (x - mu)^2 / (2 sigma^2), with sigma^2 = exp(2 log_sigma)."""
    raise NotImplementedError

In [ ]:
x = jnp.array([2.1, 1.9, 2.6, 2.4, 3.0])
fit, _ = train(gaussian_nll, {"mu": jnp.array(0.0), "log_sigma": jnp.array(0.0)}, x, lr=0.05, steps=3000)
var_hat = float(jnp.exp(2 * fit["log_sigma"]))
compare("mu vs the sample mean", fit["mu"], x.mean())
compare("variance vs np.var (divide by n)", var_hat, np.var(np.asarray(x)))
print(f"variance by gradient {var_hat:.4f} | divide by n {np.var(np.asarray(x)):.4f} | divide by n-1 {np.var(np.asarray(x), ddof=1):.4f}")
sim = np.random.default_rng(1).normal(0, 1, (100_000, 5))
print(f"mean of the MLE variance of 100000 samples of size 5 from N(0, 1): {sim.var(axis=1).mean():.3f}")

### A2. MAP for a coin: the Beta prior

The log-posterior (up to a constant) of a coin with $k$ heads in $n$ tosses under a $\text{Beta}(a, b)$ prior is
$(k + a - 1)\log\theta + (n - k + b - 1)\log(1-\theta)$. Implement `log_posterior_bernoulli(theta, k, n, a, b)`. The
driver evaluates it on a grid of $\theta$ with `jax.vmap`, takes the maximum, and compares with the closed form
$\hat\theta_{MAP} = (k + a - 1)/(n + a + b - 2)$. Then it checks that `BernoulliNB(alpha)` uses exactly this estimate
with $a = b = \alpha + 1$.

In [ ]:
def log_posterior_bernoulli(theta, k, n, a, b) -> jax.Array:
    """TODO: (k + a - 1) log(theta) + (n - k + b - 1) log(1 - theta) (a scalar theta; vmap will map it over a grid)."""
    raise NotImplementedError

In [ ]:
grid = jnp.linspace(0.0005, 0.9995, 1999)
print(" n   k  prior      grid MAP   formula")
for n, k in ((3, 3), (10, 7), (30, 21)):
    for a, b in ((1, 1), (2, 2), (10, 10)):
        lp = jax.vmap(lambda t, k=k, n=n, a=a, b=b: log_posterior_bernoulli(t, k, n, a, b))(grid)
        print(f"{n:2d}  {k:2d}  Beta({a},{b}) {float(grid[jnp.argmax(lp)]):9.4f}  {(k + a - 1) / (n + a + b - 2):8.4f}")

plot_beta_posteriors(3, 3, {"flat": (1, 1), "Laplace": (2, 2), "strong": (10, 10)})
k_word, n_spam = 0, 8                                         # a word never seen in 8 spam e-mails
Xw = np.array([[1]] * k_word + [[0]] * (n_spam - k_word) + [[1]] * 8 + [[0]] * 2)
yw = np.array([1] * n_spam + [0] * 10)
for alpha in (1.0, 3.0):
    theta_nb = np.exp(BernoulliNB(alpha=alpha).fit(Xw, yw).feature_log_prob_[1, 0])
    print(f"BernoulliNB(alpha={alpha}): P(word | spam) = {theta_nb:.4f} ; MAP with Beta({int(alpha) + 1},{int(alpha) + 1}): "
          f"{(k_word + alpha) / (n_spam + 2 * alpha):.4f}")

### A3. MAP for regression: ridge is a Gaussian prior

Linear regression with Gaussian noise $\mathcal{N}(0,\sigma^2)$ and a Gaussian prior $w_j\sim\mathcal{N}(0,\tau^2)$ (no prior
on the bias). Implement `map_loss(params, X, y, sigma2, tau2)`, the negative log-posterior without constants,
$\frac{1}{2\sigma^2}\sum_i (y_i - w^\top x_i - b)^2 + \frac{1}{2\tau^2}\lVert w\rVert^2$, and train it. Compare with
`Ridge(alpha = sigma2 / tau2)`.

In [ ]:
def map_loss(params, X, y, sigma2, tau2) -> jax.Array:
    """TODO: sum(resid^2) / (2 sigma2) + sum(w^2) / (2 tau2); the bias is not penalized."""
    raise NotImplementedError

In [ ]:
sub = np.random.default_rng(5).choice(len(ytr_h), 100, replace=False)
sigma2, tau2 = 0.4, 0.25
p, _ = train(map_loss, {"w": jnp.zeros(8), "b": jnp.zeros(())}, Xtr_h[sub], ytr_h[sub], sigma2, tau2, lr=0.05, steps=4000)
rr = Ridge(alpha=sigma2 / tau2).fit(Xtr_h[sub], ytr_h[sub])
compare("MAP weights vs Ridge", p["w"], rr.coef_)
compare("MAP bias vs Ridge", p["b"], rr.intercept_)
ols = LinearRegression().fit(Xtr_h[sub], ytr_h[sub])
print("norm of the weights: least squares", float(np.linalg.norm(ols.coef_)), "| MAP", float(jnp.linalg.norm(p["w"])))

## Part B — Naive Bayes with discrete features

$$\hat c = \arg\max_c\Big[\log P(c) + \sum_j \log P(x_j\mid c)\Big]$$

Training is **counting**. The functions take `n_classes` (2 for spam; Part D uses more).

### B1. Bernoulli Naive Bayes

Binary features $b_j = [x_j > 0]$. Implement `fit_bernoulli_nb(Xb, y, alpha, n_classes=2)` returning the log-prior of each
class and the logs of $\theta_{cj} = P(b_j=1\mid c) = (n_{cj}+\alpha)/(n_c+2\alpha)$ and of $1-\theta_{cj}$;
`joint_log_bernoulli(model, Xb)` returning $\log P(c) + \log P(x\mid c)$ for every example and class; and
`posterior_proba(joint)`, the normalized $P(c\mid x)$ computed with `logsumexp` (subtract the log-normalizer, then
`exp`).

In [ ]:
def fit_bernoulli_nb(Xb, y, alpha, n_classes=2) -> tuple[jax.Array, jax.Array, jax.Array]:
    """TODO: return (log_prior[K], log_theta[K, d], log_one_minus_theta[K, d]) with Laplace smoothing alpha."""
    raise NotImplementedError


def joint_log_bernoulli(model, Xb) -> jax.Array:
    """TODO: log P(c) + sum_j log P(b_j | c), shape (n, K). Use jnp.where and broadcasting."""
    raise NotImplementedError


def posterior_proba(joint) -> jax.Array:
    """TODO: exp(joint - logsumexp(joint, axis=1, keepdims=True))."""
    raise NotImplementedError

In [ ]:
Btr, Bte = B_s[tr_s], B_s[te_s]
bnb = fit_bernoulli_nb(Btr, ytr_s, 1.0)
bnb_sk = BernoulliNB(alpha=1.0).fit(Btr, ytr_s)
joint_b = joint_log_bernoulli(bnb, Bte)
compare("joint log-probabilities", joint_b, bnb_sk.predict_joint_log_proba(Bte))
compare("posterior probabilities", posterior_proba(joint_b)[:, 1], bnb_sk.predict_proba(Bte)[:, 1])
pred_b = np.asarray(joint_b).argmax(axis=1)
print("Bernoulli NB", {k: round(v, 3) for k, v in classification_metrics(yte_s, pred_b).items()})

# Laplace smoothing: alpha = 0 makes a class impossible when a word was never seen in it.
for n in (20, 100):
    for alpha in (0.0, 1.0):
        with np.errstate(all="ignore"):
            j = np.asarray(joint_log_bernoulli(fit_bernoulli_nb(B_s[tr_s[:n]], ytr_s[:n], alpha), Bte))
        print(f"n = {n:3d}, alpha = {alpha}: e-mails with an impossible class {int(np.isneginf(j).any(axis=1).sum()):3d} of {len(Bte)}, accuracy {np.mean(j.argmax(axis=1) == yte_s):.3f}")

## Part C — Naive Bayes with continuous features

### C1. Gaussian Naive Bayes

Implement `fit_gaussian_nb(X, y, var_smoothing=1e-9, n_classes=2)` returning the log-prior, the class means and the class
variances (the maximum-likelihood variance, **dividing by $n$**; add `var_smoothing * max_j Var(x_j)` to every variance),
and `joint_log_gaussian(model, X)`. Broadcast the examples `(n, 1, d)` against the class parameters `(1, K, d)`: no loops
over examples. Use the log-transformed features `L_s`.

In [ ]:
def fit_gaussian_nb(X, y, var_smoothing=1e-9, n_classes=2) -> tuple[jax.Array, jax.Array, jax.Array]:
    """TODO: return (log_prior[K], mean[K, d], var[K, d]); var = MLE variance + var_smoothing * max_j Var(x_j)."""
    raise NotImplementedError


def joint_log_gaussian(model, X) -> jax.Array:
    """TODO: log P(c) + sum_j log N(x_j; mean_cj, var_cj), shape (n, K). Broadcast, no loops."""
    raise NotImplementedError

In [ ]:
gnb = fit_gaussian_nb(L_s[tr_s], ytr_s)
gnb_sk = GaussianNB().fit(L_s[tr_s], ytr_s)
compare("class means", gnb[1], gnb_sk.theta_)
compare("class variances", gnb[2], gnb_sk.var_)
joint_g = joint_log_gaussian(gnb, L_s[te_s])
compare("joint log-probabilities", joint_g, gnb_sk.predict_joint_log_proba(L_s[te_s]))
pred_g = np.asarray(joint_g).argmax(axis=1)
print("Gaussian NB", {k: round(v, 3) for k, v in classification_metrics(yte_s, pred_g).items()})

### C2. Categorical Naive Bayes (discretized continuous features)

Instead of a Gaussian, cut each feature into bins and treat the bin as a category with its own probability. Implement
`fit_categorical_nb(Z, y, alpha, n_categories, n_classes=2)` (`Z` holds integer bins) returning the log-prior and
$\log\theta_{cjm} = \log\frac{n_{cjm}+\alpha}{n_{cj}+\alpha\,M}$ for class $c$, feature $j$, category $m$
($M$ = `n_categories`), and `joint_log_categorical(model, Z)`.

In [ ]:
def fit_categorical_nb(Z, y, alpha, n_categories, n_classes=2) -> tuple[jax.Array, jax.Array]:
    """TODO: return (log_prior[K], log_theta[K, d, M]) with Laplace smoothing alpha; use np.bincount."""
    raise NotImplementedError


def joint_log_categorical(model, Z) -> jax.Array:
    """TODO: log P(c) + sum_j log theta[c, j, Z[i, j]], shape (n, K)."""
    raise NotImplementedError

In [ ]:
cnb = fit_categorical_nb(Z_s[tr_s], ytr_s, 1.0, 4)
cnb_sk = CategoricalNB(alpha=1.0, min_categories=4).fit(Z_s[tr_s], ytr_s)
joint_c = joint_log_categorical(cnb, Z_s[te_s])
compare("joint log-probabilities", joint_c, cnb_sk.predict_joint_log_proba(Z_s[te_s]))
pred_c = np.asarray(joint_c).argmax(axis=1)
print("Categorical NB", {k: round(v, 3) for k, v in classification_metrics(yte_s, pred_c).items()})

# Generative or discriminative? Test accuracy against the number of training e-mails.
sizes = [20, 50, 100, 300, 1000, len(tr_s)]
curves = {"logistic": [], "Bernoulli NB": [], "Gaussian NB": [], "Categorical NB": []}
for n in sizes:
    idx = tr_s[:n]
    m_, s_ = fit_standardizer(L_s[idx])
    lr_ = LogisticRegression(C=1.0, max_iter=5000).fit(standardize(L_s[idx], m_, s_), y_s[idx])
    curves["logistic"].append(lr_.score(standardize(L_s[te_s], m_, s_), yte_s))
    curves["Bernoulli NB"].append(np.mean(np.asarray(joint_log_bernoulli(fit_bernoulli_nb(B_s[idx], y_s[idx], 1.0), Bte)).argmax(1) == yte_s))
    curves["Gaussian NB"].append(np.mean(np.asarray(joint_log_gaussian(fit_gaussian_nb(L_s[idx], y_s[idx]), L_s[te_s])).argmax(1) == yte_s))
    curves["Categorical NB"].append(np.mean(np.asarray(joint_log_categorical(fit_categorical_nb(Z_s[idx], y_s[idx], 1.0, 4), Z_s[te_s])).argmax(1) == yte_s))
plot_learning_curves(sizes, curves)
print({k: [round(float(v), 3) for v in vs] for k, vs in curves.items()})

## Part D — Naive Bayes for regression

The target is now a number. Two routes.

### D1. Discretize the target

Cut $y$ into $K$ quantile bins. Implement `bin_target(y, n_bins)` returning the interior **edges** (the quantiles at
$1/K, 2/K, \dots$: `np.quantile`) and the **bin means** $m_k$ (the mean of the training targets in each bin; the class
of a value is `np.searchsorted(edges, y, side="right")`), and `predict_expected(joint, means)`, the expected value
$\hat y = \sum_k P(k\mid x)\,m_k$ from the joint log-scores of a classifier (use `posterior_proba`).

The driver trains your Gaussian NB (Part C) on the bin labels, predicts the expected value and the most probable bin, and
compares with scikit-learn's `KBinsDiscretizer` + `GaussianNB` version.

In [ ]:
def bin_target(y, n_bins) -> tuple[np.ndarray, np.ndarray]:
    """TODO: return (interior quantile edges of y, mean of y inside each bin)."""
    raise NotImplementedError


def predict_expected(joint, means) -> np.ndarray:
    """TODO: sum_k P(k | x) m_k, with P(k | x) = posterior_proba(joint)."""
    raise NotImplementedError

In [ ]:
def report(name, pred):
    m = regression_metrics(yte_h, pred)
    print(f"{name:<40} R2 {m['r2']:.3f}  RMSE {m['rmse']:.3f}  MAE {m['mae']:.3f}")


report("mean baseline", np.full(len(yte_h), ytr_h.mean()))
lin_pred = LinearRegression().fit(Xtr_h, ytr_h).predict(Xte_h)
report("linear regression (Class 03)", lin_pred)

K = 10
edges_y, means_y = bin_target(ytr_h, K)
cls_h = np.searchsorted(edges_y, ytr_h, side="right")
joint_h = joint_log_gaussian(fit_gaussian_nb(Xtr_h, cls_h, n_classes=K), Xte_h)
pred_exp = predict_expected(joint_h, means_y)
pred_arg = means_y[np.asarray(joint_h).argmax(axis=1)]
report(f"Gaussian NB, K = {K}: expected value", pred_exp)
report(f"Gaussian NB, K = {K}: most probable bin", pred_arg)

kb = KBinsDiscretizer(K, encode="ordinal", strategy="quantile").fit(ytr_h[:, None])
cls_sk = np.asarray(kb.transform(ytr_h[:, None])).ravel().astype(int)
nb_sk = GaussianNB().fit(Xtr_h, cls_sk)
compare("bin labels vs KBinsDiscretizer", cls_h, cls_sk)
compare("expected value vs scikit-learn", pred_exp, nb_sk.predict_proba(Xte_h) @ means_y[nb_sk.classes_])
for K_ in (5, 20):
    e_, m_ = bin_target(ytr_h, K_)
    j_ = joint_log_gaussian(fit_gaussian_nb(Xtr_h, np.searchsorted(e_, ytr_h, side="right"), n_classes=K_), Xte_h)
    report(f"Gaussian NB, K = {K_}: expected value", predict_expected(j_, m_))

### D2. Discrete features for regression

Now discretize the **features** too (5 quantile bins each, `fit_bins` / `apply_bins`) and use your categorical NB with
the $K$ target bins. The driver reports the result for 3, 5, 10 and 20 bins per feature and compares with
`CategoricalNB`.

In [ ]:
def cat_reg(bins, K=10):
    edges_x = fit_bins(Xtr_h, bins)
    Ftr, Fte = apply_bins(Xtr_h, edges_x), apply_bins(Xte_h, edges_x)
    e_, m_ = bin_target(ytr_h, K)
    cls = np.searchsorted(e_, ytr_h, side="right")
    joint = joint_log_categorical(fit_categorical_nb(Ftr, cls, 1.0, bins, n_classes=K), Fte)
    sk = CategoricalNB(alpha=1.0, min_categories=bins).fit(Ftr, cls)
    return predict_expected(joint, m_), sk.predict_proba(Fte) @ m_[sk.classes_]


for bins in (3, 5, 10, 20):
    ours, theirs = cat_reg(bins)
    report(f"Categorical NB, K = 10, {bins:2d} bins per feature", ours)
    compare(f"   vs CategoricalNB ({bins} bins)", ours, theirs)

### D3. A continuous target: linear-Gaussian Naive Bayes

$$y\sim\mathcal{N}(\mu_y,\tau^2),\qquad x_j\mid y\sim\mathcal{N}(a_j + b_j\,y,\;s_j^2)\ \text{(independent given } y)$$

Implement `fit_lgnb(X, y)` returning $(\mu_y, \tau^2, a, b, s^2)$: the mean and variance of $y$, the least-squares fit of every
feature **on** $y$ ($x_j\approx a_j+b_jy$; `np.linalg.lstsq` on `[y, 1]`), and the residual variances (mean squared
residual). Then `predict_lgnb(model, X)` returning the posterior mean and standard deviation of $y$:

$$P = \frac1{\tau^2} + \sum_j \frac{b_j^2}{s_j^2}, \qquad \hat y = \frac1P\Big(\frac{\mu_y}{\tau^2} + \sum_j\frac{b_j\,(x_j-a_j)}{s_j^2}\Big),
\qquad \text{sd} = \frac1{\sqrt P}$$

The driver **verifies** your formula without it: `neg_log_posterior(y, x, model)` is written in `jax.numpy`, `jax.grad`
climbs it for a few districts, and the maximum must equal your posterior mean.

In [ ]:
def fit_lgnb(X, y) -> tuple[float, float, np.ndarray, np.ndarray, np.ndarray]:
    """TODO: return (mu_y, tau2, a, b, s2): regress every feature on y with np.linalg.lstsq([y, 1])."""
    raise NotImplementedError


def predict_lgnb(model, X) -> tuple[np.ndarray, float]:
    """TODO: return (posterior mean of y for every row, posterior standard deviation)."""
    raise NotImplementedError

In [ ]:
model = fit_lgnb(Xtr_h, ytr_h)
mean_lg, sd_lg = predict_lgnb(model, Xte_h)
report("linear-Gaussian NB (continuous target)", mean_lg)
mu_y, tau2_y, a_j, b_j, s2_j = (jnp.asarray(v) for v in model)


def neg_log_posterior(y_, x):
    return 0.5 * (y_ - mu_y) ** 2 / tau2_y + 0.5 * jnp.sum((x - a_j - b_j * y_) ** 2 / s2_j)


grad_lp = jax.jit(jax.grad(neg_log_posterior))
climbed = []
for i in range(5):
    y_ = jnp.asarray(0.0)
    for _ in range(400):
        y_ = y_ - 0.1 * grad_lp(y_, jnp.asarray(Xte_h[i]))
    climbed.append(float(y_))
compare("posterior mean vs jax.grad ascent", mean_lg[:5], climbed)
inside = np.abs(yte_h - mean_lg) < 1.645 * sd_lg
print(f"posterior sd {sd_lg:.3f} (same for every district); the 90% interval covers {inside.mean():.3f} of the test districts")
plot_pred_vs_actual(yte_h, {"linear regression": lin_pred, "Gaussian NB, K = 10": pred_exp, "linear-Gaussian NB": mean_lg})

### D4. Where the naive assumption hurts

Append $0, 1, 3, 10$ exact copies of the most informative feature (median income) to the features and refit **linear
regression** and your **linear-Gaussian NB**. The driver prints $R^2$, the posterior standard deviation and the coverage of
the 90% interval.

In [ ]:
j_inc = names_h.index("MedInc")
print("copies | R2 linear regression | R2 linear-Gaussian NB | posterior sd | 90% coverage")
for copies in (0, 1, 3, 10):
    Xa_tr = np.c_[Xtr_h, np.repeat(Xtr_h[:, [j_inc]], copies, axis=1)] if copies else Xtr_h
    Xa_te = np.c_[Xte_h, np.repeat(Xte_h[:, [j_inc]], copies, axis=1)] if copies else Xte_h
    r2_lin = regression_metrics(yte_h, LinearRegression().fit(Xa_tr, ytr_h).predict(Xa_te))["r2"]
    mean_, sd_ = predict_lgnb(fit_lgnb(Xa_tr, ytr_h), Xa_te)
    print(f"{copies:6d} | {r2_lin:20.3f} | {regression_metrics(yte_h, mean_)['r2']:21.3f} | {sd_:12.3f} | {np.mean(np.abs(yte_h - mean_) < 1.645 * sd_):.3f}")

## Part E — Probabilities and decisions

A classifier's probability is only useful if it is **calibrated**. The driver trains a logistic regression on the standardized
log features (a discriminative model) to compare with your Naive Bayes.

### E1. Log-loss, Brier score and reliability

Implement `log_loss_(y, p)` $= -\frac1n\sum[y\log p + (1-y)\log(1-p)]$ (clip $p$ to $[10^{-15}, 1-10^{-15}]$),
`brier_score(y, p)` $= \frac1n\sum(p-y)^2$ and `reliability_table(y, p, bins)`, which cuts $[0,1]$ into `bins` equal bins and
returns, for every **non-empty** bin, the mean predicted probability, the observed frequency of spam and the count.
The driver checks them against `log_loss`, `brier_score_loss` and `calibration_curve`.

In [ ]:
def log_loss_(y, p) -> float:
    """TODO: -mean(y log p + (1 - y) log(1 - p)), with p clipped to [1e-15, 1 - 1e-15]."""
    raise NotImplementedError


def brier_score(y, p) -> float:
    """TODO: mean((p - y)^2)."""
    raise NotImplementedError


def reliability_table(y, p, bins) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """TODO: for every non-empty bin of [0, 1]: (mean predicted probability, observed frequency, count)."""
    raise NotImplementedError

In [ ]:
Str_s = standardize(L_s[tr_s], *fit_standardizer(L_s[tr_s]))
Ste_s = standardize(L_s[te_s], *fit_standardizer(L_s[tr_s]))
lr = LogisticRegression(C=1.0, max_iter=5000).fit(Str_s, ytr_s)
probas = {
    "logistic": lr.predict_proba(Ste_s)[:, 1],
    "Bernoulli NB": np.asarray(posterior_proba(joint_b))[:, 1],
    "Gaussian NB": np.asarray(posterior_proba(joint_g))[:, 1],
}
for name, p in probas.items():
    print(f"{name:<13} accuracy {np.mean((p >= 0.5) == yte_s):.3f} | log-loss {log_loss_(yte_s, p):.3f} | Brier {brier_score(yte_s, p):.3f}")
p_ = probas["Gaussian NB"]
compare("log-loss vs sklearn", log_loss_(yte_s, p_), log_loss(yte_s, np.clip(p_, 1e-15, 1 - 1e-15)))
compare("Brier vs sklearn", brier_score(yte_s, p_), brier_score_loss(yte_s, p_))
frac, mean_pred = calibration_curve(yte_s, p_, n_bins=10, strategy="uniform")
mp, fr, cnt = reliability_table(yte_s, p_, 10)
compare("reliability: observed", fr, frac)
compare("reliability: predicted", mp, mean_pred)
plot_reliability(yte_s, probas)

### E2. Decisions with costs

A false positive (a real e-mail hidden) costs $c_{FP}$, a false negative (a spam delivered) costs $c_{FN}$. Implement
`bayes_threshold(c_fp, c_fn)` $= c_{FP}/(c_{FP}+c_{FN})$ (flag spam when $P(\text{spam}) >$ it) and
`expected_cost(y, p, threshold, c_fp, c_fn)`, the mean cost per e-mail. The driver compares, for the costs 1:1, 10:1 and 1:5,
the threshold 0.5, the Bayes threshold and the best threshold found by scanning the test set, for the logistic model.

In [ ]:
def bayes_threshold(c_fp, c_fn) -> float:
    """TODO: c_fp / (c_fp + c_fn)."""
    raise NotImplementedError


def expected_cost(y, p, threshold, c_fp, c_fn) -> float:
    """TODO: (c_fp * false positives + c_fn * false negatives) / n for the rule p >= threshold."""
    raise NotImplementedError

In [ ]:
p_lr = probas["logistic"]
grid_t = np.linspace(0.02, 0.99, 98)
for c_fp, c_fn in ((1, 1), (10, 1), (1, 5)):
    t_star = bayes_threshold(c_fp, c_fn)
    scan = np.array([expected_cost(yte_s, p_lr, t, c_fp, c_fn) for t in grid_t])
    print(f"FP:FN = {c_fp}:{c_fn}  t* = {t_star:.3f} | cost at 0.5: {expected_cost(yte_s, p_lr, 0.5, c_fp, c_fn):.3f} | cost at t*: "
          f"{expected_cost(yte_s, p_lr, t_star, c_fp, c_fn):.3f} | best scanned t = {grid_t[scan.argmin()]:.2f} (cost {scan.min():.3f})")
plot_cost_curve(grid_t, np.array([expected_cost(yte_s, p_lr, t, 10, 1) for t in grid_t]), bayes_threshold(10, 1))
# Naive Bayes probabilities are over-confident: the threshold computed from costs is wrong for them.
p_g = probas["Gaussian NB"]
print("Gaussian NB, costs 10:1: cost at t* =", round(expected_cost(yte_s, p_g, bayes_threshold(10, 1), 10, 1), 3),
      "| best scanned:", round(min(expected_cost(yte_s, p_g, t, 10, 1) for t in grid_t), 3))

## Part F — Challenge: Platt scaling with `jax.grad`

Repair the probabilities of the Gaussian NB. **Platt scaling** learns a logistic map of the classifier's *score* (its
log-odds $s = \log P(\text{spam}\mid x) - \log P(\text{ham}\mid x)$) to a probability: $p = \sigma(a\,s + b)$.

1. Split the training set: the first 75% fits the Gaussian NB, the last 25% is the **calibration set**.
2. The log-odds of Naive Bayes reach $\pm10^{8}$: compress them with $t = \text{sign}(s)\log(1+|s|)$ (monotone). Implement
   `platt_loss(params, t, y)` (the binary cross-entropy of $\sigma(a t + b)$, `params = {"a", "b"}`) and fit it on the
   calibration set with `train`.
3. On the test set report log-loss, Brier score and the cost at the Bayes threshold for 10:1 costs, before and after
   calibration. Compare your $(a, b)$ with a `LogisticRegression` fitted on the same one-dimensional transformed score.

In [ ]:
# TODO (home): the calibration split, platt_loss, the fit, the before/after table, and the comparison with LogisticRegression.